In [12]:
import ee
import geemap

ee.Initialize()

aoi = ee.Geometry.Rectangle([91, 25, 93, 27])

before_start = '2022-06-01'
before_end   = '2022-06-15'
after_start  = '2022-07-01'
after_end    = '2022-07-15'

collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select('VV'))

before = collection.filterDate(before_start, before_end).median().clip(aoi)
after  = collection.filterDate(after_start, after_end).median().clip(aoi)

before_filtered = before.focal_mean(30, 'circle', 'meters')
after_filtered  = after.focal_mean(30, 'circle', 'meters')

before_water = before_filtered.lt(-17)
after_water  = after_filtered.lt(-17)

flood = after_water.And(before_water.Not()).rename('flood')

waterMask = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
             .select('occurrence')
             .gt(50))

flood = flood.And(waterMask.Not())

connected = flood.connectedPixelCount(50, True)
flood_clean = flood.updateMask(connected.gte(5))

flood_clean = flood_clean.focal_max(30, 'circle', 'meters')

pixelArea = ee.Image.pixelArea()
floodAreaImage = flood_clean.multiply(pixelArea)

stats = floodAreaImage.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=30,
    maxPixels=1e10
)

flood_area_sqkm = ee.Number(stats.values().get(0)).divide(1e6)
print('Flood Area (sq km):', flood_area_sqkm.getInfo())

Map = geemap.Map()
Map.centerObject(aoi, 8)

Map.addLayer(before_filtered, {'min': -25, 'max': 0}, 'Before')
Map.addLayer(after_filtered, {'min': -25, 'max': 0}, 'After')

Map.addLayer(
    flood_clean.selfMask(),
    {'palette': ['0000FF'], 'opacity': 1},
    'Flooded Areas'
)

Map

Flood Area (sq km): 554.5121205003333


Map(center=[26.00059892469784, 92.00000000000006], controls=(WidgetControl(options=['position', 'transparent_b…